In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os

# --- CONFIGURATION ---
MODEL_PATH = "fusion_model.pth" 
IMG_SIZE = 256                  

# --- HARDWARE SETUP ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Generator hardware: {DEVICE}")

# --- 1. DEFINE THE MODEL ARCHITECTURE ---
class FusionNet(nn.Module):
    def __init__(self):
        super(FusionNet, self).__init__()
        self.enc1 = nn.Conv2d(6, 32, kernel_size=3, padding=1)
        self.enc2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=2) 
        self.enc3 = nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=2)
        self.bottle = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.dec3 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.dec2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.final = nn.Conv2d(32, 3, kernel_size=3, padding=1)
        
    def forward(self, x):
        e1 = F.relu(self.enc1(x))
        e2 = F.relu(self.enc2(e1))
        e3 = F.relu(self.enc3(e2))
        b = F.relu(self.bottle(e3))
        d3 = F.relu(self.dec3(b))
        d2 = F.relu(self.dec2(d3))
        return torch.sigmoid(self.final(d2))

# --- 2. LOAD THE TRAINED MODEL ---
model = FusionNet().to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval() 
    print("✅ AI Brain Loaded Successfully!")
else:
    print(f"❌ Error: {MODEL_PATH} not found. Train the model first.")

# --- 3. CORE WARPING & FUSION LOGIC ---
def warp_tensor(img_tensor, flow_tensor):
    N, C, H, W = img_tensor.shape
    xx = torch.arange(0, W).view(1, -1).repeat(H, 1)
    yy = torch.arange(0, H).view(-1, 1).repeat(1, W)
    xx = xx.view(1, 1, H, W).repeat(N, 1, 1, 1)
    yy = yy.view(1, 1, H, W).repeat(N, 1, 1, 1)
    grid = torch.cat((xx, yy), 1).float()
    if img_tensor.device.type != 'cpu': 
        grid = grid.to(DEVICE)
    vgrid = grid + flow_tensor
    vgrid[:, 0, :, :] = 2.0 * vgrid[:, 0, :, :] / max(W - 1, 1) - 1.0
    vgrid[:, 1, :, :] = 2.0 * vgrid[:, 1, :, :] / max(H - 1, 1) - 1.0
    vgrid = vgrid.permute(0, 2, 3, 1)
    return F.grid_sample(img_tensor, vgrid, align_corners=True)

def generate_single_middle_frame(img1_s, img2_s):
    """Generates exactly ONE intermediate frame (the 50% mark)."""
    prev_gray = cv2.cvtColor(img1_s, cv2.COLOR_BGR2GRAY)
    next_gray = cv2.cvtColor(img2_s, cv2.COLOR_BGR2GRAY)
    
    # Calculate Flow
    flow = cv2.calcOpticalFlowFarneback(prev_gray, next_gray, None, 0.5, 5, 25, 10, 5, 1.2, 0)
    
    # Prepare Tensors
    t1 = torch.from_numpy(img1_s).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    t2 = torch.from_numpy(img2_s).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    t_flow = torch.from_numpy(flow).permute(2, 0, 1).float().unsqueeze(0)
    
    t1, t2, t_flow = t1.to(DEVICE), t2.to(DEVICE), t_flow.to(DEVICE)
    
    # Run AI
    with torch.no_grad():
        warped_1 = warp_tensor(t1, t_flow * 0.5)
        warped_2 = warp_tensor(t2, t_flow * -0.5)
        cnn_input = torch.cat([warped_1, warped_2], dim=1)
        output = model(cnn_input)
        
    out_img = output.squeeze(0).permute(1, 2, 0).cpu().numpy()
    out_img = (out_img * 255).astype(np.uint8)
    
    return out_img

# --- 4. INTERACTIVE SINGLE-FRAME EXECUTION ---
print("\n--- Single Frame Interpolation Test ---")

frame_A_path = input("Enter path to FIRST keyframe: ").strip()
frame_B_path = input("Enter path to SECOND keyframe: ").strip()
output_path = input("Enter the name to save (e.g., test_middle.png): ").strip()

# --- SAFETY CHECKS ---
if not output_path.lower().endswith(('.png', '.jpg', '.jpeg')):
    print("⚠️ Automatically adding '.png' to the filename.")
    output_path += '.png'

output_dir = os.path.dirname(output_path)
if output_dir != '' and not os.path.exists(output_dir):
    os.makedirs(output_dir, exist_ok=True)
# ---------------------

if os.path.exists(frame_A_path) and os.path.exists(frame_B_path):
    print(f"\n⏳ Generating single middle frame...")
    
    # Load and resize the keyframes
    imgA = cv2.resize(cv2.imread(frame_A_path), (IMG_SIZE, IMG_SIZE))
    imgB = cv2.resize(cv2.imread(frame_B_path), (IMG_SIZE, IMG_SIZE))
    
    # Generate ONE frame
    middle_gen = generate_single_middle_frame(imgA, imgB)
    
    # Save it to disk
    cv2.imwrite(output_path, middle_gen)
    print(f"✅ Success! Saved to: '{output_path}'")
    
    # Show it on screen
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(cv2.cvtColor(imgA, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Input: Frame A")
    ax[0].axis('off')
    
    ax[1].imshow(cv2.cvtColor(middle_gen, cv2.COLOR_BGR2RGB))
    ax[1].set_title("AI Output: 50% Middle")
    ax[1].axis('off')
    
    ax[2].imshow(cv2.cvtColor(imgB, cv2.COLOR_BGR2RGB))
    ax[2].set_title("Input: Frame B")
    ax[2].axis('off')
    
    plt.tight_layout()
    plt.show()

else:
    print("\n❌ Error: Could not find one or both of your input files.")